# QEDispInv-win 测试 notebook

本 notebook 用于在同一套 demo 数据上验证 Windows 版程序的主要链路是否正常。

验证内容包括：
- `lvl-l4` 前向色散计算与核函数输出。
- `syn-nearsurface` 的前向匹配与 quick 反演。
- `syn-crustmantle` 的前向匹配与 quick 反演。
- 关键输出文件的维度、有效反演数量和核函数非零性检查。


In [1]:
from pathlib import Path
import subprocess
import sys
import numpy as np

ROOT = Path(r'E:/codes/DFSpy/FJ-QED/QEDispInv-win')
BIN = ROOT / 'bin'
DEMO = ROOT / 'demo'
print(ROOT)


E:\codes\DFSpy\FJ-QED\QEDispInv-win


In [2]:
def run_cmd(args, cwd=ROOT):
    print('>>>', ' '.join(str(x) for x in args))
    completed = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f'command failed: {completed.returncode}')
    return completed


In [3]:
run_cmd([sys.executable, str(BIN / 'forward.py'), '-c', str(DEMO / 'lvl-l4' / 'config.toml'), '-m', '0', '--compute_kernel', '-o', str(DEMO / 'lvl-l4' / 'disp_win.txt')])
disp_lvl4 = np.loadtxt(DEMO / 'lvl-l4' / 'disp_win.txt')
kernel_lvl4 = np.load(DEMO / 'lvl-l4' / 'kernel.npz')
{
    'disp_head': disp_lvl4[:5],
    'disp_shape': disp_lvl4.shape,
    'kvp_shape': kernel_lvl4['kvp'].shape,
    'kvs_max': float(np.max(np.abs(kernel_lvl4['kvs']))),
    'kvp_max': float(np.max(np.abs(kernel_lvl4['kvp']))),
    'krho_max': float(np.max(np.abs(kernel_lvl4['krho']))),
}


>>> D:\anaconda3\python.exe E:\codes\DFSpy\FJ-QED\QEDispInv-win\bin\forward.py -c E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\lvl-l4\config.toml -m 0 --compute_kernel -o E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\lvl-l4\disp_win.txt



{'disp_head': array([[0.1      , 0.5667112, 0.       ],
        [0.2      , 0.5640202, 0.       ],
        [0.3      , 0.5613623, 0.       ],
        [0.4      , 0.5587524, 0.       ],
        [0.5      , 0.5562039, 0.       ]]),
 'disp_shape': (1000, 3),
 'kvp_shape': (4, 1000),
 'kvs_max': 2.65221671750986,
 'kvp_max': 0.011286017113217413,
 'krho_max': 0.09496657453539409}

In [4]:
run_cmd([sys.executable, str(BIN / 'forward.py'), '-c', str(DEMO / 'syn-nearsurface' / 'config_win_quick.toml'), '--disp', str(DEMO / 'syn-nearsurface' / 'data.txt'), '-o', str(DEMO / 'syn-nearsurface' / 'disp_forward_win.txt')])
disp_ns = np.loadtxt(DEMO / 'syn-nearsurface' / 'disp_forward_win.txt')
disp_ns[:5]


>>> D:\anaconda3\python.exe E:\codes\DFSpy\FJ-QED\QEDispInv-win\bin\forward.py -c E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-nearsurface\config_win_quick.toml --disp E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-nearsurface\data.txt -o E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-nearsurface\disp_forward_win.txt



array([[ 6.       ,  0.4941133,  0.       ],
       [ 8.       ,  0.4258254,  0.       ],
       [10.       ,  0.3095654,  0.       ],
       [12.       ,  0.2620021,  0.       ],
       [14.       ,  0.2448138,  0.       ]])

In [5]:
run_cmd([sys.executable, str(BIN / 'inversion.py'), '-c', str(DEMO / 'syn-nearsurface' / 'config_win_quick.toml'), '-d', str(DEMO / 'syn-nearsurface' / 'data.txt'), '-o', str(DEMO / 'syn-nearsurface' / 'inv_win_full_quick.npz')])
inv_ns = np.load(DEMO / 'syn-nearsurface' / 'inv_win_full_quick.npz', allow_pickle=True)
{k: inv_ns[k].shape if hasattr(inv_ns[k], 'shape') else type(inv_ns[k]) for k in inv_ns.files}


>>> D:\anaconda3\python.exe E:\codes\DFSpy\FJ-QED\QEDispInv-win\bin\inversion.py -c E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-nearsurface\config_win_quick.toml -d E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-nearsurface\data.txt -o E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-nearsurface\inv_win_full_quick.npz



{'fitness': (1,),
 'niter': (1,),
 'z_sample': (100,),
 'vs_sample': (100,),
 'vs_hist2d': (100, 100),
 'data': (80, 3),
 'vs_mean': (100,),
 'vs_median': (100,),
 'vs_mode': (100,),
 'vs_cred10': (100,),
 'vs_cred90': (100,),
 'model_mean': (100, 5),
 'vs_ref': (100,),
 'num_init': (1,),
 'mode_used': (4,),
 'num_valid': (1,),
 'disp_syn_list': (1, 80, 3),
 'model_init_list': (1, 36, 5)}

In [6]:
run_cmd([sys.executable, str(BIN / 'forward.py'), '-c', str(DEMO / 'syn-crustmantle' / 'config_win_quick.toml'), '--disp', str(DEMO / 'syn-crustmantle' / 'data.txt'), '-o', str(DEMO / 'syn-crustmantle' / 'disp_forward_win.txt')])
disp_cm = np.loadtxt(DEMO / 'syn-crustmantle' / 'disp_forward_win.txt')
disp_cm[:5]


>>> D:\anaconda3\python.exe E:\codes\DFSpy\FJ-QED\QEDispInv-win\bin\forward.py -c E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-crustmantle\config_win_quick.toml --disp E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-crustmantle\data.txt -o E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-crustmantle\disp_forward_win.txt



array([[0.01     , 4.1784758, 0.       ],
       [0.018    , 4.0763518, 0.       ],
       [0.0259   , 3.9280519, 0.       ],
       [0.0339   , 3.7373231, 0.       ],
       [0.0418   , 3.5591093, 0.       ]])

In [7]:
run_cmd([sys.executable, str(BIN / 'inversion.py'), '-c', str(DEMO / 'syn-crustmantle' / 'config_win_quick.toml'), '-d', str(DEMO / 'syn-crustmantle' / 'data.txt'), '-o', str(DEMO / 'syn-crustmantle' / 'inv_win_full_quick.npz')])
inv_cm = np.load(DEMO / 'syn-crustmantle' / 'inv_win_full_quick.npz', allow_pickle=True)
{k: inv_cm[k].shape if hasattr(inv_cm[k], 'shape') else type(inv_cm[k]) for k in inv_cm.files}


>>> D:\anaconda3\python.exe E:\codes\DFSpy\FJ-QED\QEDispInv-win\bin\inversion.py -c E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-crustmantle\config_win_quick.toml -d E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-crustmantle\data.txt -o E:\codes\DFSpy\FJ-QED\QEDispInv-win\demo\syn-crustmantle\inv_win_full_quick.npz



{'fitness': (1,),
 'niter': (1,),
 'z_sample': (100,),
 'vs_sample': (100,),
 'vs_hist2d': (100, 100),
 'data': (125, 3),
 'vs_mean': (100,),
 'vs_median': (100,),
 'vs_mode': (100,),
 'vs_cred10': (100,),
 'vs_cred90': (100,),
 'model_mean': (100, 5),
 'vs_ref': (100,),
 'num_init': (1,),
 'mode_used': (4,),
 'num_valid': (1,),
 'disp_syn_list': (1, 125, 3),
 'model_init_list': (1, 45, 5)}

In [8]:
summary = {
    'lvl-l4_disp_rows': int(disp_lvl4.shape[0]),
    'lvl-l4_kernel_nonzero': bool(np.max(np.abs(kernel_lvl4['kvs'])) > 0.0),
    'syn-nearsurface_forward_rows': int(disp_ns.shape[0]),
    'syn-crustmantle_forward_rows': int(disp_cm.shape[0]),
    'syn-nearsurface_inv_valid': int(inv_ns['num_valid'][0]),
    'syn-crustmantle_inv_valid': int(inv_cm['num_valid'][0]),
}
summary


{'lvl-l4_disp_rows': 1000,
 'lvl-l4_kernel_nonzero': True,
 'syn-nearsurface_forward_rows': 80,
 'syn-crustmantle_forward_rows': 125,
 'syn-nearsurface_inv_valid': 1,
 'syn-crustmantle_inv_valid': 1}